# 提交说明（Chapter 4）

本 notebook 对应“主数据富化与类别对齐（Panel）”机制实现。

- 使用参数单元管理输入/输出路径。
- 提交前请确认不包含内部路径和敏感数据。

In [ ]:
# 参数区（提交版本）
import os
from pathlib import Path

DATA_ROOT = Path(os.environ.get("CH5_DATA_ROOT", "./data_sample"))
OUTPUT_ROOT = Path(os.environ.get("CH5_OUTPUT_ROOT", "./outputs"))

for p in [DATA_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
# Global Cateogry Mapping for IQVIA
# Eric Wu, GM Office, BCH China
# Jul,2024

In [ ]:
import pandas as pd
import difflib

In [ ]:
def modify_skuid(df, column_name='SKUID'):
    """
    修改DataFrame中SKUID列：在倒数第三位插入0，然后左侧补0至12位，保持字符型。
    
    参数:
    df (pandas.DataFrame): 输入的DataFrame
    column_name (str): 包含SKUID的列名，默认为'SKUID'
    
    返回:
    pandas.DataFrame: 处理后的DataFrame
    """
    # 确保列存在
    if column_name not in df.columns:
        raise ValueError(f"列 '{column_name}' 不存在于DataFrame中")
    
    # 将SKUID转换为字符串
    df[column_name] = df[column_name].astype(str)
    
    # 在倒数第三位插入0
    df[column_name] = df[column_name].apply(
        lambda x: x[:-2] + '0' + x[-2:] if len(x) >= 2 else x + '0' * (3 - len(x))
    )
    
    # 左侧补0至12位
    df[column_name] = df[column_name].str.zfill(12)
    
    return df

In [ ]:
path_input = r'//bcnshgs0295/02_FlatDataSource/01_UploadFile/82_GM_Office/02_Data_Platform/05_IQVIA/'
path_output = r'//bcnshgs0295/02_FlatDataSource/01_UploadFile/82_GM_Office/02_Data_Platform/05_IQVIA/'

In [ ]:
df_global_mapping_reference = pd.read_excel(path_input + 'IQVIA_Global_Category_Mapping_Historical.xlsx')
df_property_reference = pd.read_excel(path_input + '牌子表_CHC_with CHC info.xlsx')

In [ ]:
#df_property_reference['pfc'] = df_property_reference['pfc'].astype(str).str.zfill(12)
#df_global_mapping_reference = modify_skuid(df_global_mapping_reference)

In [ ]:
df_property_reference.rename(columns={'pfc': 'SKUID'}, inplace=True)
print (df_global_mapping_reference.shape[0])

In [ ]:
df_global_mapping_reference = pd.merge(df_global_mapping_reference, df_property_reference, 
                             on = 'SKUID',
                             how='left')

In [ ]:
df_global_mapping_new_SKU = df_property_reference[df_property_reference["update vs 上期牌子表"] == "新增"]
df_global_mapping_new_SKU.shape

In [ ]:
skus_in_reference = df_global_mapping_new_SKU['SKUID'].isin(df_global_mapping_reference['SKUID'])
df_global_mapping_new_SKU_in_reference = df_global_mapping_new_SKU[skus_in_reference]

print("Existing SKUID:")
print(df_global_mapping_new_SKU_in_reference['SKUID'])
print(df_global_mapping_new_SKU_in_reference['SKUID'].tolist())

In [ ]:
new_skus_not_in_reference = ~df_global_mapping_new_SKU['SKUID'].isin(df_global_mapping_reference['SKUID'])
df_global_mapping_new_SKU_unique = df_global_mapping_new_SKU[new_skus_not_in_reference]

#df_global_mapping_new_SKU_unique_filtered = df_global_mapping_new_SKU_unique[df_global_mapping_new_SKU_unique['e6'] == "WM" || 
#                                                                             df_global_mapping_new_SKU_unique['e6'] == "HF"]

df_global_mapping_new_SKU_unique_filtered = df_global_mapping_new_SKU_unique[df_global_mapping_new_SKU_unique['e6'].isin(["WM", "HEALTHFOOD"])]

new_skuid_count = len(df_global_mapping_new_SKU_unique_filtered)
print(f"New Add {new_skuid_count} Non-TCM SKU to df_global_mapping_reference")
df_global_mapping = pd.concat([df_global_mapping_reference, df_global_mapping_new_SKU_unique_filtered], ignore_index=True)
df_global_mapping.reset_index(drop=True, inplace=True)

In [ ]:
merged_df = df_global_mapping.copy()
merged_df['New SKU'] = False
merged_df['Comments'] = ""

for index, row in merged_df.iterrows():
    if pd.isnull(row['New Global Category']):

        conditions = pd.Series([True] * len(df_global_mapping_reference), index=df_global_mapping_reference.index)
        

        condition_fields = [
            'CHC III SHORT DESC',
            'CHC II SHORT DESC',
            'CHC IV SHORT DESC',
            'e9',
            'e8',
            'e7',
            'e6',
            'e2',
            'e1',
            'l1',
            'code'
        ]
        

        for field in condition_fields:

            conditions = conditions & (df_global_mapping_reference[field] == row[field])
            filtered_df = df_global_mapping_reference[conditions]
            

            unique_combinations = filtered_df.drop_duplicates(subset=['New Global Category', 'New Global Sub-Category', 'New Global Segment'])
            
            if len(unique_combinations) == 1:

                merged_df.at[index, 'New Global Category'] = unique_combinations.iloc[0]['New Global Category']
                merged_df.at[index, 'New Global Sub-Category'] = unique_combinations.iloc[0]['New Global Sub-Category']
                merged_df.at[index, 'New Global Segment'] = unique_combinations.iloc[0]['New Global Segment']
                merged_df.at[index, 'New SKU'] = True
                merged_df.at[index, 'Comments'] = "Conditional Match"
                break
        

        if len(unique_combinations) != 1:
            merged_df.at[index, 'New Global Category'] = 'to be mapped'
            merged_df.at[index, 'New Global Sub-Category'] = 'to be mapped'
            merged_df.at[index, 'New Global Segment'] = 'to be mapped'
            merged_df.at[index, 'New SKU'] = True

In [ ]:
to_be_mapped_count = (merged_df['New Global Category'] == 'to be mapped').sum()
to_be_mapped_count

In [ ]:
to_be_mapped_rows = merged_df['New Global Category'] == 'to be mapped'

for index, row in merged_df[to_be_mapped_rows].iterrows():
    same_category_rows = (
    
    (df_global_mapping_reference['CHC III SHORT DESC'] == row['CHC III SHORT DESC']) &
    (df_global_mapping_reference['CHC IV SHORT DESC'] == row['CHC IV SHORT DESC']) &
    (df_global_mapping_reference['e9'] == row['e9']) &
    (df_global_mapping_reference['e8'] == row['e8']) &
    (df_global_mapping_reference['SKUID'] != row['SKUID'])
            
    )
    
    if same_category_rows.any():
        candidate_names = df_global_mapping_reference[same_category_rows]['gene']
        closest_match = difflib.get_close_matches(row['gene'], candidate_names, n=1, cutoff=0.5)
        
        if closest_match:
            match_index = df_global_mapping_reference[same_category_rows & (df_global_mapping_reference['gene'] == closest_match[0])].index[0]
            
            merged_df.at[index, 'New Global Category'] = df_global_mapping_reference.at[match_index, 'New Global Category']
            merged_df.at[index, 'New Global Sub-Category'] = df_global_mapping_reference.at[match_index, 'New Global Sub-Category']
            merged_df.at[index, 'New Global Segment'] = df_global_mapping_reference.at[match_index, 'New Global Segment']
            merged_df.at[index, 'Comments'] = "AI Match"

In [ ]:
merged_df.loc[(merged_df['New SKU'] == True) & (merged_df['Comments'] == ""), 'Comments'] = "to be mapped manually"

In [ ]:
to_be_mapped_count = (merged_df['New Global Category'] == 'to be mapped').sum()
to_be_mapped_count

In [ ]:
merged_df.to_excel(path_output + 'IQVIA_Global_Category_Mapping_in_Process.xlsx')

In [ ]:
### END